# Train the multimodal dementia model on Kaggle (2×T4)

**Workflow:** train here across **both GPUs** → download `best_model.pt` → run inference locally.

Setup before running:
- **Settings → Accelerator → GPU T4 ×2**
- **Settings → Internet → On** (needed to clone the repo + download HuBERT/BERT)
- **Add Data →** search `dementia-detection-using-speech` (by *tahouramorovati*) and add it. It mounts read-only under `/kaggle/input/...`.

Sections: (1) get code, (2) install, (3) confirm 2 GPUs, (4) locate data, (5) sanity pass, (6) parallel train, (7) download `.pt`.

## 1. Get the code

In [ ]:
import os
REPO_URL = "https://github.com/VeNoM-hubs/Neuralyzer.git"
%cd /kaggle/working
if not os.path.exists("Neuralyzer"):
    !git clone $REPO_URL
%cd Neuralyzer
!git pull  # make sure we have train_parallel.py + latest fixes

## 2. Install dependencies
Kaggle already ships CUDA `torch`/`torchaudio`, so we install everything else.

In [ ]:
!pip install -q "transformers>=4.40,<5" "scikit-learn>=1.3.0" "PyYAML>=6.0" "tqdm>=4.65.0" "soundfile>=0.12.1"

## 3. Confirm both GPUs are visible
You should see **2** devices. If it says 1, set **Accelerator → GPU T4 ×2** and restart the session.

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
n = torch.cuda.device_count()
print("GPUs visible:", n)
for i in range(n):
    print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")
assert n >= 2, "Enable 'GPU T4 x2' in Settings for parallel training."

## 4. Locate the data

The **Add Data** panel mounts the dataset read-only under `/kaggle/input/...`. No `kaggle.json` needed. The cell below auto-detects the folder that contains `Archive/`.

In [ ]:
import glob, os
archives = glob.glob('/kaggle/input/**/Archive', recursive=True)
assert archives, "No 'Archive/' found under /kaggle/input — add the dataset via 'Add Data'."
DATA_ROOT = os.path.dirname(archives[0])
print("DATA_ROOT =", DATA_ROOT)
print("record folders:", len(glob.glob(os.path.join(DATA_ROOT, 'Archive', 'Process-rec-*'))))
print("CTD wavs:", len(glob.glob(os.path.join(DATA_ROOT, 'Archive', '**', '*__CTD.wav'), recursive=True)))

## 5. Quick sanity pass (1 seed, 3 epochs, single GPU)
Do this first to confirm the pipeline runs end-to-end before committing to the full run.

In [ ]:
!python train.py --dataset process --data-root {DATA_ROOT} \
    --output-dir /kaggle/working/sanity --single-seed --max-epochs 3

## 6. Full run — both GPUs in parallel

`train_parallel.py` launches one `train.py` process per GPU (pinned via `CUDA_VISIBLE_DEVICES`), splits the seeds across them, warms the HuggingFace cache once to avoid a download race, streams both logs live with `[gpuN]` prefixes, and prints the combined mean ± std at the end.

MINE's per-run batch stays intact — we parallelize **seeds**, not the model. Each T4 has a full 16 GB, so `--max-chunks 6 --batch-size 12` is comfortable; drop them if you hit OOM.

In [ ]:
!python train_parallel.py --dataset process --data-root {DATA_ROOT} \
    --output-dir /kaggle/working/outputs \
    --seeds 42,43,44,45,46 \
    --max-chunks 6 --batch-size 12

## 7. Pick the best checkpoint & download

Each GPU writes `outputs/gpuN/best_model.pt` (weights + config bundled). The cell picks the highest-F1 one across GPUs from the per-GPU `summary.csv` files and copies it to `/kaggle/working/best_model.pt` for download (right-click in the Output panel → Download).

In [ ]:
import csv, glob, os, shutil

best = {"f1": -1.0, "ckpt": None, "seed": None}
for summ in glob.glob('/kaggle/working/outputs/gpu*/summary.csv'):
    gpu_dir = os.path.dirname(summ)
    with open(summ, newline='') as f:
        for row in csv.DictReader(f):
            f1 = float(row.get('f1', -1))
            if f1 > best['f1']:
                best = {"f1": f1, "ckpt": os.path.join(gpu_dir, 'best_model.pt'),
                        "seed": row.get('seed')}

assert best['ckpt'] and os.path.exists(best['ckpt']), "No best_model.pt found — did training finish?"
dst = '/kaggle/working/best_model.pt'
shutil.copyfile(best['ckpt'], dst)
print(f"Best: seed {best['seed']}, F1={best['f1']:.4f}")
print(f"Copied {best['ckpt']} -> {dst} ({round(os.path.getsize(dst)/1e6, 1)} MB)")
print("Download it from the Kaggle Output panel (right side).")